# Foundry IQ knowledge base with Work IQ


This notebook creates a Foundry IQ knowledge base that includes a Work IQ knowledge source that connects to Microsoft 365 (Outlook, Teams, Calendar, OneDrive). The KB queries it at runtime to surface emails, chats, and calendar events relevant to the user's question.
This notebook uses `InteractiveBrowserCredential` to sign in the current user and obtain a delegated token for Azure AI Search. Search uses that delegated user identity when querying protected sources such as Work IQ.

## Step 1: Set up environment variables

Load the configuration for your Azure AI Search and Azure OpenAI resources. The `.env` file in the project folder has the values needed for this part.

In [1]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

Environment variables loaded


## Step 2: Login and acquire delegated token

In order for knowledge base retrieval to use your identity with Work IQ, you need an OAuth token for a signed-in user who has access to Microsoft 365 data in this tenant.

Run the cell below to sign in to Azure using the credentials in the instructions sidebar. If successful, it acquires a delegated token for the Azure AI Search scope and saves it for use as `query_source_authorization` during retrieval.

In [2]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=AZURE_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

/Users/pamelafox/iqdeepdive-foundryiq-1/.venv/lib/python3.14/site-packages/msal/oauth2cli/oauth2.py:453: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


Acquired token for logged-in user


## Step 3: Create three knowledge sources

For this knowledge base, you'll create three knowledge sources: the same two search-index sources used earlier, plus a Work IQ knowledge source.

* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `workiq-knowledge-source`: Points to Work IQ for workplace context

In [3]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Contoso Work IQ knowledge source",
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)
print("Knowledge sources created")


Knowledge sources created


## Step 4: Create the multi-source + Work IQ knowledge base

A knowledge base combines the knowledge sources with an LLM and instructions for how retrieval should behave.

For this notebook, the knowledge base uses an `outputMode` of `answerSynthesis` so the attached model can answer the question directly. It also uses a `retrievalReasoningEffort` of `low` so the model can help with query planning and source selection.

In [4]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-workiq-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining indexed company documents and Work IQ",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for workplace context (emails, chats, events, meetings, etc). Use the search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


Knowledge base created


## Step 5: Query the knowledge base

Ask a two-part question: one half needs your actual inbox context from Work IQ, the other needs company policy from the HR index.

* _"What are my recent emails about the Professional Claw Hammer stock issue at Seattle?"_ → should come from **Work IQ**
* _"Which Contoso job roles are responsible for inventory strategy and budget oversight?"_ → should come from `hrdocs`

> ⏳ Expect to wait 1-2 minutes for the response, as Work IQ increases retrieval latency. The retrieval request specifies `max_runtime_in_seconds` to increase the time the knowledge base waits for the results from the sources.

In [5]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("Check my recent work emails for messages about the Professional Claw Hammer. "
            "Summarize what colleagues are saying and what actions have been requested. " 
            "Which Contoso job roles are responsible for inventory strategy and budget oversight?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))


I couldn’t find any recent work email messages about the Professional Claw Hammer in the available workplace results.[ref_id:0]

For the Contoso roles tied to **inventory strategy**, the clearest match is **Director of Operations**, which is explicitly responsible for developing and implementing strategies to manage inventory and stock levels.[ref_id:1]

For **budget oversight**, several roles are explicitly responsible:
- **Chief Financial Officer (CFO)** oversees budgeting, financial planning and analysis, forecasting, and financial reporting.[ref_id:0]
- **Director of Operations** manages the budget and financial operations of the company.[ref_id:1]
- **Senior Manager of Operations** develops, implements, and monitors operational plans and budgets.[ref_id:3]
- **Vice President of Operations** oversees the development and implementation of operational plans and budgets.[ref_id:4]
- **Chief Operating Officer (COO)** manages the budget and ensures the organization is operating within its means.[ref_id:9]
- **Manager of Operations** develops and manages operational budgets and monitors financial performance.[ref_id:11]

If you want the most direct answer to your final question, the best matches are:
- **Inventory strategy:** **Director of Operations**.[ref_id:1]
- **Budget oversight:** **CFO** for organization-wide financial oversight.[ref_id:0] **COO**, **VP of Operations**, **Director of Operations**, **Senior Manager of Operations**, and **Manager of Operations** also carry explicit budget responsibilities in operations contexts.[ref_id:9][ref_id:4][ref_id:1][ref_id:3][ref_id:11]

I did not find any email-specific colleague commentary or requested actions about the Professional Claw Hammer in the available results.[ref_id:0]

### Review the activity log

For this activity log, you will see a "workIQ" step for the call made to Work IQ, along with "searchIndex" steps for each query sent to a search index.

In [6]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "inputTokens": 1425,
    "outputTokens": 112,
    "modelName": "gpt-5.4",
    "elapsedMs": 1826
  },
  {
    "type": "workIQ",
    "id": 1,
    "knowledgeSourceName": "workiq-knowledge-source",
    "queryTime": "2026-07-14T21:09:01.8957591Z",
    "count": 0,
    "error": {
      "message": "WorkIQ A2A call failed with status code BadRequest."
    },
    "workIQArguments": {
      "search": "Check my recent work emails for messages about the Professional Claw Hammer. Summarize what colleagues are saying and what actions have been requested. Which Contoso job roles are responsible for inventory strategy and budget oversight?"
    }
  },
  {
    "type": "searchIndex",
    "id": 2,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-07-14T21:09:01.9511356Z",
    "count": 28,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "inventory strategy role responsibility",
      "filter": null,
      

### Review the references

For the Work IQ knowledge source, the references have `type: "workIQ"`. The `sourceData` contains `extracts` with a synthesized answer from the Work IQ agent, and each reference contains `attributions` with URLs that link to any referenced emails, documents, etc.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

Look at the references printed above. Can you identify:

1. Which knowledge source answered the **email thread summary** question?
2. Which knowledge source answered the **job roles** question?

## ✅ Mission complete

**What you built:**

* ✓ **Work IQ Knowledge Source**: A source that connects your real Microsoft 365 data to the knowledge base, so the KB can retrieve emails, Teams messages, and calendar context at query time.
* ✓ **3-source Knowledge Base**: A single KB spanning HR docs, health docs, and live M365 workplace data, with the agent routing each sub-question to the right source.
* ✓ **Hybrid inbox and policy retrieval**: Work IQ surfaced the email thread about the Seattle stock outage. The HR index answered the budget management question.